# Team02 FastAPI + ngrok Relay for SageMaker Serverless Endpoint

This notebook creates a temporary **FastAPI relay inside the Team02 SageMaker Studio/Jupyter environment** and exposes that relay through **ngrok**.

## Recommended demo architecture

```text
Local / external Streamlit app
        ↓ HTTPS
ngrok public URL
        ↓
FastAPI relay running inside Team02 SageMaker Studio
        ↓ boto3 using the SageMaker execution role
AWS SageMaker Serverless Endpoint
team02-diabetes-risk-test
```

### Why this design is preferred

The Streamlit computer does **not** need AWS Access Key / Secret Access Key credentials.

The FastAPI relay runs inside SageMaker Studio/Jupyter, where `boto3` can use the notebook's SageMaker execution role to invoke the deployed endpoint.

The Streamlit app only needs:

1. the ngrok `/predict` URL; and
2. the temporary **Relay API Key** generated by this notebook.

> **Important:** The Relay API Key is **not obtained from AWS**. It is a temporary shared secret created by this notebook for the demo.

> Use this relay only for temporary project testing/demo. Stop ngrok and FastAPI after the demonstration.

## Security correction for final submission

The relay shared secret must never appear in saved notebook output, source examples or screenshots.

This corrected notebook:
- reads the secret from `RELAY_API_KEY` or prompts with `getpass`;
- never prints the secret value;
- uses placeholders in PowerShell/external-caller examples;
- writes public-relay evidence without credentials;
- keeps AWS credentials only inside the SageMaker environment.

**Rotate the previously exposed temporary relay secret before using this corrected notebook.**


## 1. Confirm that this notebook is running with an AWS identity

Run this notebook **inside the Team02 SageMaker Studio/Jupyter environment**.

If this cell succeeds, the relay can use the attached SageMaker execution role.  
If it fails with `NoCredentialsError`, you are probably running the notebook outside AWS and should move the relay back into SageMaker Studio.

In [ ]:
import boto3
import json
from botocore.exceptions import NoCredentialsError, ClientError

try:
    sts = boto3.client("sts")
    identity = sts.get_caller_identity()
    print("AWS identity available:")
    print(json.dumps(identity, indent=2))
except NoCredentialsError:
    raise RuntimeError(
        "No AWS credentials are available. Run this relay notebook inside "
        "the Team02 SageMaker Studio/Jupyter environment."
    )

## 2. Team02 configuration

The endpoint shown in the deployment evidence is:

- **Region:** `ap-southeast-1`
- **Endpoint:** `team02-diabetes-risk-test`
- **Type:** SageMaker Serverless
- **Expected status:** `InService`

A strong temporary Relay API Key is generated for the current notebook session.

Do not put this key in GitHub or in the final report.

In [ ]:
import os
from getpass import getpass

REGION = "ap-southeast-1"
ENDPOINT_NAME = "team02-diabetes-risk-test"

# Shared secret between Streamlit and the FastAPI relay.
# NEVER print or save the value in notebook output.
RELAY_API_KEY = os.getenv("RELAY_API_KEY", "").strip()

if not RELAY_API_KEY:
    RELAY_API_KEY = getpass(
        "Paste the temporary relay shared secret (input hidden): "
    ).strip()

if len(RELAY_API_KEY) < 24:
    raise ValueError(
        "Use a temporary relay secret of at least 24 characters."
    )

print("Region   :", REGION)
print("Endpoint :", ENDPOINT_NAME)
print("Relay API key configured securely: YES (value not displayed)")


## 3. Confirm the deployed SageMaker endpoint is available

This verifies that the Team02 SageMaker execution role can see the endpoint and that it is ready before we start the relay.

In [ ]:
import boto3
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name=REGION)

try:
    endpoint = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint status:", endpoint["EndpointStatus"])
    print("Endpoint ARN:", endpoint["EndpointArn"])
    print("Creation time:", endpoint.get("CreationTime"))
    print("Last modified:", endpoint.get("LastModifiedTime"))

    if endpoint["EndpointStatus"] != "InService":
        print(
            "WARNING: Endpoint is not InService. "
            "Prediction tests may fail until deployment is ready."
        )
except ClientError as e:
    print("Could not describe endpoint.")
    print("Code:", e.response["Error"]["Code"])
    print("Message:", e.response["Error"]["Message"])
    raise

## 4. Define the Team02 diabetes input contract

The deployed Team02 endpoint expects **21 BRFSS-derived input features**, not the heart-disease fields from the original Team40 relay notebook.

The feature order/names used here are:

```text
HighBP
HighChol
CholCheck
BMI
Smoker
Stroke
HeartDiseaseorAttack
PhysActivity
Fruits
Veggies
HvyAlcoholConsump
AnyHealthcare
NoDocbcCost
GenHlth
MentHlth
PhysHlth
DiffWalk
Sex
Age
Education
Income
```

The sample below is only a technical test record.

In [ ]:
FEATURE_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]

sample_payload = {
    "HighBP": 0,
    "HighChol": 0,
    "CholCheck": 1,
    "BMI": 25.0,
    "Smoker": 0,
    "Stroke": 0,
    "HeartDiseaseorAttack": 0,
    "PhysActivity": 1,
    "Fruits": 1,
    "Veggies": 1,
    "HvyAlcoholConsump": 0,
    "AnyHealthcare": 1,
    "NoDocbcCost": 0,
    "GenHlth": 2,
    "MentHlth": 0,
    "PhysHlth": 0,
    "DiffWalk": 0,
    "Sex": 1,
    "Age": 8,
    "Education": 5,
    "Income": 6,
}

assert list(sample_payload.keys()) == FEATURE_COLUMNS
print("Payload contains", len(sample_payload), "features.")
print(json.dumps(sample_payload, indent=2))

## 5. Test direct endpoint invocation from SageMaker Studio

This is an important diagnostic step.

If this succeeds, then:

```text
SageMaker notebook role → SageMaker Runtime → endpoint
```

is working before FastAPI/ngrok is introduced.

In [ ]:
import boto3
import json
from botocore.exceptions import ClientError, NoCredentialsError

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

try:
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(sample_payload).encode("utf-8"),
    )

    raw_result = response["Body"].read().decode("utf-8")
    print("Raw endpoint response:")
    print(raw_result)

    try:
        parsed_result = json.loads(raw_result)
        print("\nParsed endpoint response:")
        print(json.dumps(parsed_result, indent=2))
    except json.JSONDecodeError:
        print("Endpoint returned non-JSON content.")

except NoCredentialsError:
    raise RuntimeError(
        "AWS credentials are not available. Run this notebook inside "
        "the Team02 SageMaker environment."
    )
except ClientError as e:
    print("Endpoint invocation failed.")
    print("Code:", e.response["Error"]["Code"])
    print("Message:", e.response["Error"]["Message"])
    raise

## 6. Install relay dependencies

Run this once in the SageMaker notebook environment.

In [ ]:
%pip install -q fastapi "uvicorn[standard]" pyngrok requests "pydantic>=2"

## 7. Create the Team02 FastAPI relay

The relay exposes:

- `GET /` — basic service metadata;
- `GET /health` — authenticated health check that also checks SageMaker endpoint status;
- `POST /predict` — authenticated prediction relay;
- `GET /docs` — FastAPI interactive documentation.

### Security controls

- The external caller cannot choose an arbitrary endpoint.
- The endpoint name is fixed server-side.
- `/health` and `/predict` require `x-api-key`.
- The payload is validated against the 21 expected features.
- Unexpected additional fields are rejected.
- The relay returns model output but does not expose AWS credentials.

In [ ]:
%%writefile relay_api.py

import hmac
import json
import os
from typing import Any

import boto3
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError, NoCredentialsError
from fastapi import Depends, FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import APIKeyHeader
from pydantic import BaseModel, ConfigDict, Field

REGION = os.environ.get("AWS_REGION", "ap-southeast-1")
ENDPOINT_NAME = os.environ.get(
    "ENDPOINT_NAME",
    "team02-diabetes-risk-test",
)
RELAY_API_KEY = os.environ.get("RELAY_API_KEY", "")

if not RELAY_API_KEY:
    raise RuntimeError(
        "RELAY_API_KEY is missing. Set it before starting the FastAPI relay."
    )

app = FastAPI(
    title="Team02 Diabetes SageMaker Relay",
    version="1.0",
    description=(
        "Authenticated FastAPI relay for the Team02 ITI113 diabetes "
        "screening-support SageMaker endpoint."
    ),
)

# Temporary demo setting. Streamlit normally calls the relay server-side,
# so CORS is not required for that path. Keep this broad only for demo use.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["Content-Type", "x-api-key"],
)

api_key_header = APIKeyHeader(name="x-api-key", auto_error=False)

runtime = boto3.client(
    "sagemaker-runtime",
    region_name=REGION,
    config=Config(
        connect_timeout=10,
        read_timeout=70,
        retries={"max_attempts": 2, "mode": "standard"},
    ),
)

sagemaker = boto3.client(
    "sagemaker",
    region_name=REGION,
    config=Config(
        connect_timeout=10,
        read_timeout=20,
        retries={"max_attempts": 2, "mode": "standard"},
    ),
)


class DiabetesFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")

    HighBP: int = Field(ge=0, le=1)
    HighChol: int = Field(ge=0, le=1)
    CholCheck: int = Field(ge=0, le=1)
    BMI: float = Field(gt=0, le=100)
    Smoker: int = Field(ge=0, le=1)
    Stroke: int = Field(ge=0, le=1)
    HeartDiseaseorAttack: int = Field(ge=0, le=1)
    PhysActivity: int = Field(ge=0, le=1)
    Fruits: int = Field(ge=0, le=1)
    Veggies: int = Field(ge=0, le=1)
    HvyAlcoholConsump: int = Field(ge=0, le=1)
    AnyHealthcare: int = Field(ge=0, le=1)
    NoDocbcCost: int = Field(ge=0, le=1)
    GenHlth: int = Field(ge=1, le=5)
    MentHlth: int = Field(ge=0, le=30)
    PhysHlth: int = Field(ge=0, le=30)
    DiffWalk: int = Field(ge=0, le=1)
    Sex: int = Field(ge=0, le=1)
    Age: int = Field(ge=1, le=13)
    Education: int = Field(ge=1, le=6)
    Income: int = Field(ge=1, le=8)


def require_api_key(
    supplied_key: str | None = Depends(api_key_header),
) -> None:
    if supplied_key is None or not hmac.compare_digest(
        supplied_key,
        RELAY_API_KEY,
    ):
        raise HTTPException(status_code=401, detail="Invalid relay API key.")


@app.get("/")
def home() -> dict[str, Any]:
    return {
        "status": "running",
        "service": "Team02 Diabetes SageMaker Relay",
        "endpoint": ENDPOINT_NAME,
        "region": REGION,
        "routes": [
            "GET /health",
            "POST /predict",
            "GET /docs",
        ],
    }


@app.get("/health")
def health(_: None = Depends(require_api_key)) -> dict[str, Any]:
    try:
        desc = sagemaker.describe_endpoint(EndpointName=ENDPOINT_NAME)
        return {
            "relay": "ok",
            "endpoint_name": ENDPOINT_NAME,
            "endpoint_status": desc.get("EndpointStatus"),
            "region": REGION,
        }

    except NoCredentialsError:
        raise HTTPException(
            status_code=503,
            detail=(
                "AWS credentials are not available to the relay. "
                "Run this relay inside the Team02 SageMaker Studio/Jupyter environment."
            ),
        )

    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code", "ClientError")
        message = exc.response.get("Error", {}).get("Message", "")
        raise HTTPException(
            status_code=502,
            detail=f"SageMaker health check failed: {code}: {message}",
        )


@app.post("/predict")
def predict(
    features: DiabetesFeatures,
    _: None = Depends(require_api_key),
) -> Any:
    payload = features.model_dump()

    try:
        response = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Accept="application/json",
            Body=json.dumps(payload).encode("utf-8"),
        )

        raw = response["Body"].read().decode("utf-8")

        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            raise HTTPException(
                status_code=502,
                detail="SageMaker returned a non-JSON response.",
            )

    except NoCredentialsError:
        raise HTTPException(
            status_code=503,
            detail=(
                "AWS credentials are not available to the relay. "
                "Run this relay inside the Team02 SageMaker Studio/Jupyter environment."
            ),
        )

    except (BotoCoreError, ClientError) as exc:
        if isinstance(exc, ClientError):
            code = exc.response.get("Error", {}).get("Code", "ClientError")
            message = exc.response.get("Error", {}).get("Message", "")
        else:
            code = type(exc).__name__
            message = str(exc)

        raise HTTPException(
            status_code=502,
            detail=f"SageMaker invocation failed: {code}: {message}",
        )

## 8. Start the FastAPI relay inside SageMaker

The server listens on port `8000`.

The notebook passes the Team02 endpoint configuration and temporary Relay API Key through environment variables.

In [ ]:
import os
import subprocess
import time

os.environ["AWS_REGION"] = REGION
os.environ["ENDPOINT_NAME"] = ENDPOINT_NAME
os.environ["RELAY_API_KEY"] = RELAY_API_KEY

# Stop a previous relay process if this cell is re-run.
try:
    relay_process.terminate()
    relay_process.wait(timeout=5)
    print("Stopped previous FastAPI server.")
except Exception:
    pass

relay_process = subprocess.Popen(
    [
        "uvicorn",
        "relay_api:app",
        "--host", "0.0.0.0",
        "--port", "8000",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(3)

if relay_process.poll() is not None:
    print("FastAPI failed to start. Output:")
    print(relay_process.stdout.read())
    raise RuntimeError("FastAPI relay did not start.")

print("FastAPI relay started.")
print("PID:", relay_process.pid)
print("Local root:   http://127.0.0.1:8000/")
print("Local health: http://127.0.0.1:8000/health")
print("Local docs:   http://127.0.0.1:8000/docs")
print("Local predict:http://127.0.0.1:8000/predict")

## 9. Test the local relay before using ngrok

Test `/health` first.  
A successful response should report:

```json
{
  "relay": "ok",
  "endpoint_name": "team02-diabetes-risk-test",
  "endpoint_status": "InService",
  "region": "ap-southeast-1"
}
```

In [ ]:
import requests
import json

local_health_url = "http://127.0.0.1:8000/health"

headers = {
    "x-api-key": RELAY_API_KEY,
    "Content-Type": "application/json",
}

response = requests.get(
    local_health_url,
    headers=headers,
    timeout=30,
)

print("Status code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)

response.raise_for_status()

### Test local `/predict`

This verifies the complete path:

```text
FastAPI → boto3 → SageMaker endpoint → FastAPI
```

In [ ]:
local_predict_url = "http://127.0.0.1:8000/predict"

response = requests.post(
    local_predict_url,
    headers=headers,
    json=sample_payload,
    timeout=75,
)

print("Status code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)

response.raise_for_status()

## 10. Configure ngrok

Use your ngrok **authtoken** here.

The ngrok authtoken and Relay API Key are two different credentials:

| Credential | Purpose |
|---|---|
| ngrok authtoken | Allows this notebook to create an ngrok tunnel |
| Relay API Key | Protects `/health` and `/predict` from unauthorised callers |

The Relay API Key is generated in Section 2 and is **not supplied by AWS**.

`getpass()` prevents the ngrok token from appearing directly in the notebook cell.

In [ ]:
from getpass import getpass
from pyngrok import ngrok

NGROK_AUTH_TOKEN = getpass("Paste ngrok auth token: ").strip()

if not NGROK_AUTH_TOKEN:
    raise ValueError("No ngrok auth token supplied.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok auth token configured for this notebook session.")

## 11. Start the public ngrok tunnel

Only the temporary FastAPI relay on port `8000` is exposed.

The Streamlit application will call:

```text
https://<ngrok-domain>/predict
```

In [ ]:
from pyngrok import ngrok
import time

print("Stopping any ngrok process managed by this notebook...")

try:
    ngrok.kill()
except Exception as e:
    print("No local ngrok process to stop:", e)

time.sleep(3)

print("Starting fresh ngrok tunnel on FastAPI port 8000...")

public_tunnel = ngrok.connect(
    addr=8000,
    proto="http"
)

public_url = public_tunnel.public_url.rstrip("/")

RELAY_HEALTH_URL = public_url + "/health"
RELAY_PREDICT_URL = public_url + "/predict"

print("=" * 72)
print("NGROK TUNNEL STARTED")
print("=" * 72)
print("Public relay root:")
print(public_url)

print("\nHealth URL:")
print(RELAY_HEALTH_URL)

print("\nPrediction URL:")
print(RELAY_PREDICT_URL)

print("\nFastAPI docs:")
print(public_url + "/docs")
print("=" * 72)

## 12. Test the public ngrok relay

This simulates the same request that the Streamlit application will make from outside SageMaker.

In [ ]:
headers = {
    "x-api-key": RELAY_API_KEY,
    "Content-Type": "application/json",
}

print("Testing public health route...")
health_response = requests.get(
    RELAY_HEALTH_URL,
    headers=headers,
    timeout=30,
)

print("Health status:", health_response.status_code)
try:
    print(json.dumps(health_response.json(), indent=2))
except Exception:
    print(health_response.text)

health_response.raise_for_status()

print("\nTesting public prediction route...")
predict_response = requests.post(
    RELAY_PREDICT_URL,
    headers=headers,
    json=sample_payload,
    timeout=75,
)

print("Prediction status:", predict_response.status_code)
try:
    print(json.dumps(predict_response.json(), indent=2))
except Exception:
    print(predict_response.text)

predict_response.raise_for_status()

# Save sanitized public-relay evidence. No secret is written.
from datetime import datetime, timezone
from urllib.parse import urlsplit
from pathlib import Path

public_result = predict_response.json()
sanitized_relay_host = urlsplit(RELAY_PREDICT_URL).netloc

relay_evidence = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "relay_host": sanitized_relay_host,
    "health_http_status": health_response.status_code,
    "prediction_http_status": predict_response.status_code,
    "sagemaker_endpoint": ENDPOINT_NAME,
    "region": REGION,
    "prediction_response": public_result,
    "credential_handling": (
        "x-api-key used at runtime; secret value intentionally not recorded"
    ),
}

Path("team02_public_relay_evidence.json").write_text(
    json.dumps(relay_evidence, indent=2, default=str),
    encoding="utf-8",
)
print("\nSaved sanitized evidence: team02_public_relay_evidence.json")


## 13. Values to enter in the Team02 Streamlit app

Your Streamlit application should use:

- **Relay predict URL:** the ngrok URL ending in `/predict`
- **Relay API Key:** the temporary key generated in Section 2

It should **not** use AWS Access Keys.

Run this cell and copy the two values into Streamlit.

In [ ]:
print("=" * 72)
print("STREAMLIT CONNECTION SETTINGS")
print("=" * 72)
print("Relay Predict URL:")
print(RELAY_PREDICT_URL)
print()
print("Relay API Key:")
print("<use the SAME temporary secret supplied securely in Section 2>")
print()
print("Security note: the secret is intentionally not printed by this notebook.")
print("=" * 72)


### Optional: configure the Streamlit app through PowerShell environment variables

If you are running `team02_streamlit_app.py` on your Windows computer, use:

```powershell
$env:RELAY_PREDICT_URL="<COPY THE NGROK /predict URL FROM THIS NOTEBOOK>"
$env:RELAY_API_KEY="<COPY THE RELAY API KEY FROM THIS NOTEBOOK>"

streamlit run team02_streamlit_app.py
```

Notice that there is **no local AWS credential configuration** in this architecture.

The AWS call is made by the relay inside SageMaker Studio.

In [ ]:
powershell_example = f'''$env:RELAY_PREDICT_URL="{RELAY_PREDICT_URL}"
$env:RELAY_API_KEY="<PASTE_THE_SAME_TEMPORARY_SECRET>"

streamlit run team02_streamlit_app.py
'''

print(powershell_example)


## 14. External caller example

The external UI needs only:

- Relay `/predict` URL
- Relay API Key
- Correct 21-feature JSON payload

It does not need the AWS account ID, IAM role, Access Key ID, Secret Access Key, or Session Token.

In [ ]:
example_code = f'''import requests

url = "{RELAY_PREDICT_URL}"

headers = {{
    "x-api-key": "<PASTE_THE_SAME_TEMPORARY_SECRET>",
    "Content-Type": "application/json",
}}

payload = {json.dumps(sample_payload, indent=4)}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=75,
)

print(response.status_code)
print(response.json())
'''

print(example_code)


## 15. Demo architecture evidence

For the ITI113 Final Report, this notebook demonstrates the backend deployment path:

```text
Streamlit client
  ↓ HTTPS / x-api-key
ngrok
  ↓
FastAPI authenticated relay
  ↓ boto3 / SageMaker execution role
SageMaker Serverless Endpoint
`team02-diabetes-risk-test`
  ↓
XGBoost screening-support prediction
```

### Demonstrated backend evidence

- fixed SageMaker endpoint integration;
- serverless endpoint availability check;
- direct endpoint invocation test;
- authenticated FastAPI relay;
- strict 21-feature request validation;
- local and public relay prediction tests;
- separation of external client credentials from AWS credentials;
- sanitized `team02_public_relay_evidence.json` with no secret;
- documented shutdown procedure.

### Final UI evidence still required

Run the corrected Streamlit app, make one successful prediction, download `team02_streamlit_e2e_evidence.json`, and capture one screenshot showing the result. This proves:

**Streamlit → FastAPI relay → SageMaker endpoint → response → Streamlit UI**

Do not fabricate or manually edit that evidence.

### AI governance implications

- The client does not receive AWS credentials.
- The relay cannot invoke arbitrary endpoint names supplied by the client.
- Input schema validation reduces malformed/out-of-domain requests.
- The Streamlit application states that the output is **screening support, not a diagnosis**.
- Formal fairness, explainability, robustness and subgroup evidence remain documented separately in the Final Report/Test Report.


## 16. Stop ngrok and FastAPI after the demo

Run this when testing/presentation is complete.

This reduces unnecessary public exposure and avoids accidental endpoint calls.

In [ ]:
# Stop ngrok tunnels created by pyngrok.
try:
    ngrok.kill()
    print("ngrok tunnels stopped.")
except Exception as e:
    print("ngrok stop issue:", e)

# Stop FastAPI relay.
try:
    relay_process.terminate()
    relay_process.wait(timeout=5)
    print("FastAPI relay stopped.")
except Exception as e:
    print("FastAPI stop issue:", e)

# Troubleshooting

## Streamlit says:
`{"detail":"AWS credentials are not available to the relay."}`

This means the FastAPI relay itself is running somewhere that does not have AWS credentials.

### Recommended fix

Run **this relay notebook inside SageMaker Studio/Jupyter**, then expose the relay through ngrok.

Your Windows Streamlit client should call the ngrok URL.  
It should not run the SageMaker relay locally unless your Windows machine has separately configured AWS credentials.

---

## `401 Invalid relay API key`

The Relay API Key is the temporary value generated by this notebook in **Section 2**.

It does not come from AWS.

Copy the exact same value into the Streamlit app.

---

## `AccessDeniedException`

The SageMaker execution role has AWS credentials but does not have sufficient permission to invoke the endpoint.

The role needs permission similar to:

```json
{
  "Effect": "Allow",
  "Action": [
    "sagemaker:InvokeEndpoint",
    "sagemaker:DescribeEndpoint"
  ],
  "Resource": "arn:aws:sagemaker:ap-southeast-1:044528205969:endpoint/team02-diabetes-risk-test"
}
```

The exact permission policy should follow the permissions available for the Team02 AWS environment.

---

## `ValidationException`, HTTP 422, missing fields or extra fields

Ensure the client sends exactly the 21 Team02 diabetes features.

Do not send the old Team40 heart-disease fields such as:

```text
cp
trestbps
chol
thalach
oldpeak
ca
thal
```

Those belong to the original relay prototype and are not the deployed Team02 diabetes input contract.

---

## Endpoint is not `InService`

Check:

```python
sm.describe_endpoint(
    EndpointName="team02-diabetes-risk-test"
)
```

Wait until `EndpointStatus` is `InService`.

---

## ngrok URL changes

A temporary/free ngrok URL may change when the tunnel is restarted.

Whenever the ngrok URL changes, update `RELAY_PREDICT_URL` in Streamlit.

The Relay API Key may also change if Section 2 is re-run because a new temporary secret is generated.

---

## ngrok token security

Do not hard-code your ngrok authtoken into the notebook before submitting it.

Use `getpass()` as implemented above. If a token has been exposed in chat, Git, screenshots, or submitted files, rotate it in your ngrok account.

---

## Final demo sequence

For a clean presentation:

1. Open this notebook in Team02 SageMaker Studio/Jupyter.
2. Run Sections 1–5 and confirm direct SageMaker invocation succeeds.
3. Start FastAPI.
4. Test local `/health` and `/predict`.
5. Start ngrok.
6. Test public `/health` and `/predict`.
7. Copy the public `/predict` URL and Relay API Key into Streamlit.
8. Open Streamlit and click **Test relay connection**.
9. Submit a screening record.
10. Show the prediction result and deployment metadata.
11. Stop ngrok and FastAPI after the demo.